## Snake visualization demo (no pygame window required)

This notebook reconstructs a `(C, H, W)` grid from `SnakeEnv` **flat observations**, then renders an animation with matplotlib.

- Works even without a GUI (e.g., on Slurm) because it does not open a pygame window.
- To visualize a trained policy, set `CKPT_PATH` to `save_models/<run_name>/best_ppo.pt`.


In [2]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

# Make imports work from the project root (or current working dir)
PROJECT_DIR = os.getcwd()
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from envs.pythonGame import SnakeEnv, SnakeConfig


pygame 2.6.1 (SDL 2.28.4, Python 3.10.18)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
def obs_flat_to_grid(obs_flat: np.ndarray, grid_size: int = 36, include_walls_channel: bool = True):
    """Convert flat obs (1, obs_dim) or (obs_dim,) to grid (C, H, W)."""
    v = obs_flat
    if v.ndim == 2:
        v = v[0]
    C = 4 if include_walls_channel else 3
    return v.reshape(C, grid_size, grid_size)


def grid_to_rgb(grid: np.ndarray):
    """grid: (C,H,W), channels: 0=head 1=body 2=apple 3=walls(optional)"""
    C, H, W = grid.shape
    rgb = np.zeros((H, W, 3), dtype=np.float32)

    head = grid[0] > 0.5
    body = grid[1] > 0.5
    apple = grid[2] > 0.5
    walls = (grid[3] > 0.5) if C >= 4 else np.zeros((H, W), dtype=bool)

    # Layering: walls -> apple -> body -> head
    rgb[walls] = (1.0, 0.2, 0.2)
    rgb[apple] = (0.2, 0.4, 1.0)
    rgb[body] = (0.2, 0.8, 0.2)
    rgb[head] = (0.1, 1.0, 0.1)
    return rgb


def animate_frames(frames, fps: int = 10, title: str = ""):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_title(title)
    ax.axis("off")
    im = ax.imshow(frames[0], interpolation="nearest")

    def _update(i):
        im.set_data(frames[i])
        return (im,)

    ani = animation.FuncAnimation(
        fig, _update, frames=len(frames), interval=int(1000 / fps), blit=True
    )
    plt.close(fig)
    return HTML(ani.to_jshtml())


## 1) Random policy visualization (sanity check)


In [4]:
cfg = SnakeConfig(
    obs_mode="flat",
    fps=10,
    max_steps=300,
    include_walls_channel=True,
)

env = SnakeEnv(cfg=cfg, render_mode=None, multi_agent=True)


def random_policy(_obs):
    a = np.random.randint(0, 4)
    return np.array([a], dtype=np.int64)


obs, info = env.reset(seed=0)
frames = []
done = False
last_info = info

while not done:
    grid = obs_flat_to_grid(
        obs, grid_size=cfg.grid_size, include_walls_channel=cfg.include_walls_channel
    )
    frames.append(grid_to_rgb(grid))
    obs, rew, done, last_info = env.step(random_policy(obs))

display(
    animate_frames(
        frames,
        fps=cfg.fps,
        title=f"random | score={last_info.get('score')} steps={last_info.get('steps')}",
    )
)


## 2) Visualize a trained policy (load `best_ppo.pt`)

Note: your current training uses **flat observations** (e.g., 5184 dims for 4×36×36), so we run the policy on flat obs and reshape it back to a grid for visualization.


In [7]:
import torch
from models import ActorDiscrete

# TODO: set this to your checkpoint path
CKPT_PATH = "save_models/ppo_snake_transformer_0306_142849_seed0/best_ppo.pt"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Flat obs dimension: C*H*W (default C=4, H=W=36 => 5184)
OBS_DIM = (4 if cfg.include_walls_channel else 3) * cfg.grid_size * cfg.grid_size
ACT_DIM = 4
HIDDEN_SIZE = 128  # must match training config: actor_hidden_size

actor = ActorDiscrete(
    obs_dim=OBS_DIM,
    a_size=ACT_DIM,
    hidden_size=HIDDEN_SIZE,
    device=str(device),
).to(device)

ckpt = torch.load(CKPT_PATH, map_location=device)
actor.load_state_dict(ckpt["policy"])
actor.eval()


def greedy_policy(obs):
    # obs: (1, obs_dim)
    obs_t = torch.from_numpy(obs).float().to(device)
    logits = actor(obs_t)
    a = int(torch.argmax(logits, dim=-1).item())
    return np.array([a], dtype=np.int64)


obs, info = env.reset(seed=0)
frames = []
done = False
last_info = info

while not done:
    grid = obs_flat_to_grid(
        obs, grid_size=cfg.grid_size, include_walls_channel=cfg.include_walls_channel
    )
    frames.append(grid_to_rgb(grid))
    obs, rew, done, last_info = env.step(greedy_policy(obs))

display(
    animate_frames(
        frames,
        fps=cfg.fps,
        title=f"policy | score={last_info.get('score')} steps={last_info.get('steps')}",
    )
)
